# NeuroGolf Task 006 Baseline
This notebook trains a baseline model for NeuroGolf task 006 and exports it to ONNX.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "onnx_tool", "onnxscript"])

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import json
import os

# Find the exact dataset path in Kaggle dynamically
KAGGLE_INPUT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'task006.json' in files:
        KAGGLE_INPUT = root
        break

if KAGGLE_INPUT is None:
    raise FileNotFoundError("Dataset not found in /kaggle/input")

print(f"Using dataset path: {KAGGLE_INPUT}")

UTILS_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'neurogolf_utils.py' in files:
        UTILS_DIR = root
        break

if UTILS_DIR:
    sys.path.append(UTILS_DIR)

import neurogolf_utils

class BaselineModel(nn.Module):
    def __init__(self, channels=10):
        super(BaselineModel, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, channels, kernel_size=1)
        )
    
    def forward(self, x):
        return self.net(x)

def train_task(task_id):
    with open(os.path.join(KAGGLE_INPUT, f"task{task_id:03d}.json"), "r") as f:
        task_data = json.load(f)
    
    all_examples = task_data["train"] + task_data["test"] + task_data.get("arc-gen", [])
    
    X_train = []
    Y_train = []
    
    for ex in all_examples:
        bench = neurogolf_utils.convert_to_numpy(ex)
        if bench:
            X_train.append(bench["input"])
            output_labels = np.argmax(bench["output"].squeeze(0), axis=0)
            Y_train.append(output_labels)
    
    X_train = torch.tensor(np.array(X_train)).squeeze(1)
    Y_train = torch.tensor(np.array(Y_train)).long()
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model = BaselineModel().to(device)
    X_train = X_train.to(device)
    Y_train = Y_train.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    print(f"Training Task {task_id} with {len(X_train)} examples...")
    for epoch in range(1000):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, Y_train)
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 100 == 0:
            preds = torch.argmax(outputs, dim=1)
            correct = (preds == Y_train).all().item()
            print(f"Epoch {epoch+1}, Loss: {loss.item():.6f}, Perfect Match: {correct}")
            if correct and loss.item() < 0.0001:
                break
    
    model.eval()
    model.cpu()
    dummy_input = torch.randn(1, 10, 30, 30)
    onnx_path = f"task{task_id:03d}.onnx"
    torch.onnx.export(model, dummy_input, onnx_path, 
                      input_names=['input'], output_names=['output'],
                      dynamic_axes=None)
    print(f"Model exported to {onnx_path}")
    
    import onnx
    onnx_model = onnx.load(onnx_path)
    neurogolf_utils.verify_network(onnx_model, task_id, task_data)

train_task(6)
